In [2]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import sys
import os
sys.path.append(os.path.abspath('C:/Users/Grant/OneDrive/Occami Analytica/Sports Analytics/NBA'))
import db_connection as connector
import matplotlib.pyplot as plt

In [4]:
player_df = pd.read_csv('PlayerDataWithPCAs.csv')
print(player_df.head())

   player  team  boxscore  minutes_played  field_goals  field_goal_attempts  \
0       1    23        29       35.333333     5.666667            15.000000   
1       1    23        53       35.333333     5.666667            14.000000   
2       1    23        58       35.666667     7.000000            16.000000   
3       1    23        72       38.000000     8.000000            16.000000   
4       1    23        90       37.333333     6.333333            15.333333   

   three_pointers  three_point_attempts  free_throws  free_throw_attempts  \
0        1.000000              1.666667     4.333333                    4   
1        0.666667              1.333333     3.333333                    3   
2        0.333333              1.666667     2.333333                    2   
3        0.333333              2.000000     2.333333                    4   
4        0.333333              2.000000     2.333333                    4   

   ...    blocks  turnovers     fouls     points       PC1    

In [30]:
def get_team_stats_for_game(team, boxscore, days=3):

    data = {'team': [],
            'boxscore': [],
            'opponent_field_goals': [],
            'opponent_field_goal_attempts': [],
            'opponent_three_pointers': [],
            'opponent_three_point_attempts': [],
            'opponent_free_throws': [],
            'opponent_free_throw_attempts': [],
            'opponent_offensive_rebounds': [],
            'opponent_defensive_rebounds': [],
            'opponent_total_rebounds': [],
            'opponent_assists': [],
            'opponent_steals': [],
            'opponent_blocks': [],
            'opponent_turnovers': [],
            'opponent_fouls': [],
            'opponent_points': []}

    with connector.connect_to_database() as conn:
        cursor = conn.cursor()
        sql_statement = f"SELECT * FROM team_stats WHERE boxscore = {boxscore} and team != {team}"
        cursor.execute(sql_statement)
        game = cursor.fetchall()[0]

        opponent = game[1]

        sql_statement = f'SELECT * FROM team_stats WHERE team={opponent} and boxscore < {boxscore}'

        cursor.execute(sql_statement)
        previous_games = cursor.fetchall()

        if len(previous_games) < days:
            df = None
        else:
            important_games = previous_games[-days:]

            for i in range(len(important_games)):
                game = important_games[i]

                data['team'].append(team)
                data['boxscore'].append(game[2])
                data['opponent_field_goals'].append(game[3])
                data['opponent_field_goal_attempts'].append(game[4])
                data['opponent_three_pointers'].append(game[5])
                data['opponent_three_point_attempts'].append(game[6])
                data['opponent_free_throws'].append(game[7])
                data['opponent_free_throw_attempts'].append(game[8])
                data['opponent_offensive_rebounds'].append(game[9])
                data['opponent_defensive_rebounds'].append(game[10])
                data['opponent_total_rebounds'].append(game[11])
                data['opponent_assists'].append(game[12])
                data['opponent_steals'].append(game[13])
                data['opponent_blocks'].append(game[14])
                data['opponent_turnovers'].append(game[15])
                data['opponent_fouls'].append(game[16])
                data['opponent_points'].append(game[17])

    df = pd.DataFrame(data)

    columns_to_average = ['opponent_field_goals', 'opponent_field_goal_attempts', 
                          'opponent_three_pointers', 'opponent_three_point_attempts',
                          'opponent_free_throws', 'opponent_free_throw_attempts',
                          'opponent_offensive_rebounds','opponent_defensive_rebounds',
                          'opponent_total_rebounds', 'opponent_assists',
                          'opponent_steals', 'opponent_blocks',
                          'opponent_turnovers', 'opponent_fouls', 'opponent_points']
    
    df[columns_to_average] = df[columns_to_average].rolling(window=days).mean()

    return df.dropna()

In [ ]:
dataframes = []
days=3

for i in range(len(player_df)):

    data = player_df.iloc[i]

    team = int(data['team'])
    boxscore = int(data['boxscore'])

    df = get_team_stats_for_game(team, boxscore, days)

    if not df.empty:
        dataframes.append(df)
